In [1]:
!pip install -q torch==2.5.1

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchvision 0.26.0+cu130 requires torch==2.11.0, but you have torch 2.5.1 which is incompatible.


In [2]:
!pip install -q torchvision==0.21.0

In [3]:
!pip install -q accelerate==1.2.1

In [4]:
!pip install -q tokenizers==0.21.0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.20.2 requires huggingface-hub<3.0,>=1.31.0, but you have huggingface-hub 0.36.2 which is incompatible.
transformers 5.20.2 requires tokenizers<0.24.0,>=0.23.1, but you have tokenizers 0.21.0 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.41.0 requires huggingface-hub<2.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [5]:
!pip install -q safetensors==0.4.1

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
accelerate 1.2.1 requires safetensors>=0.4.3, but you have safetensors 0.4.1 which is incompatible.
transformers 5.20.2 requires huggingface-hub<3.0,>=1.31.0, but you have huggingface-hub 0.36.2 which is incompatible.
transformers 5.20.2 requires safetensors>=0.8.0, but you have safetensors 0.4.1 which is incompatible.
transformers 5.20.2 requires tokenizers<0.24.0,>=0.23.1, but you have tokenizers 0.21.0 which is incompatible.
diffusers 0.41.0 requires huggingface-hub<2.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.41.0 requires safetensors>=0.8.0, but you have safetensors 0.4.1 which is incompatible.


In [6]:
!pip install -q datasets==2.21.0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2024.6.1 which is incompatible.


In [7]:
!pip install -q evaluate

# Data Loading

# Data Preprocessing

In [1]:
from datasets import load_dataset

dataset_restaurant = load_dataset("jakartaresearch/semeval-absa", name='restaurant')
dataset_laptop = load_dataset("jakartaresearch/semeval-absa", name='laptop')
dataset_fabsa = load_dataset("jordiclive/FABSA")

Generating train split:   0%|          | 0/3044 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/800 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/3048 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/800 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/7930 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1057 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1587 [00:00<?, ? examples/s]

In [2]:
!pip install -q transformers==4.48.2 huggingface_hub==0.28.1

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
accelerate 1.2.1 requires safetensors>=0.4.3, but you have safetensors 0.4.1 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.28.1 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.28.1 which is incompatible.
diffusers 0.41.0 requires huggingface-hub<2.0,>=1.32.0, but you have huggingface-hub 0.28.1 which is incompatible.
diffusers 0.41.0 requires safetensors>=0.8.0, but you have safetensors 0.4.1 which is incompatible.


## Expand Dataset into a convenient format

In [3]:
from tqdm.auto import tqdm
from typing import TypedDict, Generator
from datasets import Dataset, Features, Value, ClassLabel, DatasetDict


class LabeledExample(TypedDict):
    aspect: str
    labels: str
    sentence: str


def expand_dataset_semeval(dataset: Dataset) -> Generator[LabeledExample, None, None]:
    POLARITY_SET = {'positive', 'negative', 'neutral'}
    for data in tqdm(dataset):
        sentence = data['text']  # type: ignore
        terms = data['aspects']['term']  # type: ignore
        polarities = data['aspects']['polarity']  # type: ignore

        for term, polarity in zip(terms, polarities):
            if polarity not in POLARITY_SET or term == '':
                continue
            yield {
                'aspect': term,
                'labels': polarity,
                'sentence': sentence
            }


def get_dataset_semeval(dataset: DatasetDict) -> DatasetDict:
    ds_features = Features({
        'aspect': Value('string'),
        'labels': ClassLabel(names=['positive', 'negative', 'neutral']),
        'sentence': Value('string')
    })

    ds = DatasetDict(
        {
            split_name: \
            Dataset.from_generator(
                expand_dataset_semeval,
                features=ds_features,
                gen_kwargs={'dataset': dataset[split_name]}
            )
            for split_name in ['train', 'validation']
        }
    )
    return ds


def expand_dataset_fabsa(dataset: Dataset) -> Generator[LabeledExample, None, None]:
    POLARITY_SET = {'positive', 'negative', 'neutral'}
    for data in tqdm(dataset):
        sentence = data['text']  # type: ignore

        for term, polarity in data['labels']:
            if polarity not in POLARITY_SET or term == '':
                continue
            yield {
                'aspect': term.split(": ")[-1],
                'labels': polarity,
                'sentence': sentence
            }


def get_dataset_fabsa(dataset: DatasetDict) -> DatasetDict:
    ds_features = Features({
        'aspect': Value('string'),
        'labels': ClassLabel(names=['positive', 'negative', 'neutral']),
        'sentence': Value('string')
    })

    ds = DatasetDict(
        {
            split_name: \
            Dataset.from_generator(
                expand_dataset_fabsa,
                features=ds_features,
                gen_kwargs={'dataset': dataset[split_name]}
            )
            for split_name in ['train', 'validation', 'test']
        }
    )
    return ds


## Tokenize

In [4]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_name = "microsoft/deberta-v3-base" # 'google-bert/bert-base-cased'
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)  # Must be like BertTokenizer
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
# Load and preprocess datasets into raw (non-tokenized) format
ds_restaurant_raw = get_dataset_semeval(dataset_restaurant)
ds_laptop_raw = get_dataset_semeval(dataset_laptop)
ds_fabsa_raw = get_dataset_fabsa(dataset_fabsa)

Generating train split: 0 examples [00:00, ? examples/s]

  0%|          | 0/3044 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

  0%|          | 0/800 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

  0%|          | 0/3048 [00:00<?, ?it/s]

... (5 outputs left out)

Generating train split: 0 examples [00:00, ? examples/s]

  0%|          | 0/1057 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

  0%|          | 0/1587 [00:00<?, ?it/s]

In [6]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm  # for progress bar
import numpy as np

# Load a pre-trained sentence transformer model for augmentation
augmentation_model = SentenceTransformer('all-mpnet-base-v2')  # Best for semantic similarity

def augment_dataset(dataset, model):
    augmented_data = []
    for example in tqdm(dataset):
        sentence = example['sentence']
        aspect = example['aspect']

        # Compute embeddings
        sentence_embedding = model.encode(sentence)
        aspect_embedding = model.encode(aspect)

        # Compute cosine similarity and take its absolute value (modulus)
        similarity = abs(cosine_similarity([sentence_embedding], [aspect_embedding])[0][0])

        # If the aspect is unrelated (mod of similarity < threshold), replace the label with neutral
        if similarity < 0.0001:  # Adjust threshold as needed
            example['labels'] = 2  # Replace label with neutral

        # Add the example to the augmented dataset
        augmented_data.append({
            'aspect': aspect,
            'labels': example['labels'],
            'sentence': sentence
        })

    return Dataset.from_list(augmented_data)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
# Augment all datasets
augmented_restaurant_train = augment_dataset(ds_restaurant_raw['train'], augmentation_model)

100%|██████████| 3608/3608 [01:35<00:00, 37.78it/s]


In [8]:
augmented_laptop_train = augment_dataset(ds_laptop_raw['train'], augmentation_model)


100%|██████████| 2328/2328 [01:04<00:00, 36.30it/s]


In [9]:
augmented_fabsa_train = augment_dataset(ds_fabsa_raw['train'], augmentation_model)

100%|██████████| 13998/13998 [06:11<00:00, 37.63it/s]


In [10]:
from datasets import concatenate_datasets
# Combine augmented training datasets
train_dataset_raw = concatenate_datasets([
    augmented_restaurant_train,
    augmented_laptop_train,
    augmented_fabsa_train
])

In [11]:
def tokenize_function(examples):
    """This tokenizer returns the format -
    [CLS] aspect tokens [SEP] sentence tokens [SEP]
    """
    return tokenizer(examples['aspect'], examples['sentence'], padding=True, truncation=True, return_token_type_ids=False, max_length=512)

In [12]:
# Tokenize the combined augmented training dataset
train_dataset = train_dataset_raw.map(tokenize_function, batched=True)

Map:   0%|          | 0/19934 [00:00<?, ? examples/s]

Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pai

In [13]:
ds_restaurant_val = ds_restaurant_raw['validation'].map(tokenize_function, batched=True)
ds_laptop_val = ds_laptop_raw['validation'].map(tokenize_function, batched=True)
ds_fabsa_test = ds_fabsa_raw['test'].map(tokenize_function, batched=True)

Map:   0%|          | 0/1120 [00:00<?, ? examples/s]

Map:   0%|          | 0/638 [00:00<?, ? examples/s]

Map:   0%|          | 0/2812 [00:00<?, ? examples/s]

Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.


# Training

In [ ]:
from huggingface_hub import login
login("hf_XXXX")  # Replace with your actual Hugging Face token

In [15]:
repo_id = "Surabhii/deberta-v3-base-absa-semeval2014-fabsa-v2"

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
api.create_repo(
    repo_id=repo_id,
    repo_type="model",
    private=True,
    exist_ok=True,
    token = "hf_XXXX"  # Replace with your actual Hugging Face token
)

RepoUrl('https://huggingface.co/Surabhii/deberta-v3-base-absa-semeval2014-fabsa-v2', endpoint='https://huggingface.co', repo_type='model', repo_id='Surabhii/deberta-v3-base-absa-semeval2014-fabsa-v2')

In [28]:
from huggingface_hub import login, whoami
from getpass import getpass

login(token=getpass("Enter your new Hugging Face write token: "))
print("Logged-in account:", whoami()["name"])

Enter your new Hugging Face write token: ··········
Logged-in account: Surabhii


In [ ]:
from transformers import (
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from datasets import concatenate_datasets

training_args = TrainingArguments(
    output_dir="./results",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir="./logs",
    learning_rate=5e-5,
    report_to="none",

    save_strategy="steps",
    save_steps=500,
    save_total_limit=None,

    push_to_hub=True,
    hub_model_id=repo_id,
    hub_strategy="checkpoint",
)

# # Freeze all the parameters in the model
# for param in model.deberta.parameters():
#     param.requires_grad = False

# # Enable gradients for the classifier layer (last layer)
# for param in model.classifier.parameters():
#     param.requires_grad = True

import numpy as np
from sklearn.utils.class_weight import compute_class_weight
import torch
from torch import nn
from sklearn.metrics import precision_recall_fscore_support



CLASS_WEIGHTS = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_dataset['labels']),
    y=train_dataset['labels']
)

In [42]:


class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        # Forward pass
        outputs = model(**inputs)
        logits = outputs.get('logits')

        # Define fixed weights for positive, negative, and neutral classes
        fixed_weights = torch.tensor([1.0, 1.0, 2]).to(logits.device).float()

        # Compute custom loss with fixed weights
        loss_fct = nn.CrossEntropyLoss(weight=fixed_weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


In [43]:
trainer = CustomTrainer(
    model=model,                         # the instantiated 🤗 Transformers model to be trained
    args=training_args,                  # training arguments, defined above
    train_dataset=train_dataset,
    data_collator=DataCollatorWithPadding(tokenizer, padding=True, max_length=512)
)

In [44]:
model.train()
trainer.train()

/usr/local/lib/python3.13/dist-packages/transformers/tokenization_utils_base.py:2699: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


Step,Training Loss
500,0.638800
1000,0.520000
1500,0.462800
2000,0.401600
2500,0.400600
3000,0.310000
3500,0.314500
4000,0.292500
4500,0.315000
5000,0.266500


/usr/local/lib/python3.13/dist-packages/transformers/tokenization_utils_base.py:2699: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/tokenization_utils_base.py:2699: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/tokenization_utils_base.py:2699: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/tokenization_utils_base.py:2699: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
/usr

Step,Training Loss
500,0.638800
1000,0.520000
1500,0.462800
2000,0.401600
2500,0.400600
3000,0.310000
3500,0.314500
4000,0.292500
4500,0.315000
5000,0.266500


/usr/local/lib/python3.13/dist-packages/transformers/tokenization_utils_base.py:2699: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


TrainOutput(global_step=7476, training_loss=0.3219180681158222, metrics={'train_runtime': 8701.915, 'train_samples_per_second': 6.872, 'train_steps_per_second': 0.859, 'total_flos': 1.512708511278744e+16, 'train_loss': 0.3219180681158222, 'epoch': 3.0})

# Evaluation

In [45]:
import evaluate
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)  # Ensure predictions are derived correctly

    # Load the f1 metric from evaluate
    f1 = evaluate.load('f1')

    # Calculate F1 score using the 'macro' average method
    f1_result = f1.compute(predictions=predictions, references=labels, average='macro')

    # Compute the confusion matrix
    conf_matrix = confusion_matrix(labels, predictions)

    # Return a dictionary with the metric name and its value
    return {"f1_score": f1_result['f1'], "confusion_matrix": conf_matrix, 'classification_report': classification_report(labels, predictions,  digits=4)}

In [46]:
model.eval()
pred = trainer.predict(ds_restaurant_val)
restaurant_metrics = compute_metrics(
    (pred.predictions, ds_restaurant_val['labels']))
print(restaurant_metrics['classification_report'])

/usr/local/lib/python3.13/dist-packages/transformers/tokenization_utils_base.py:2699: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


              precision    recall  f1-score   support

           0     0.9258    0.9602    0.9427       728
           1     0.8462    0.8418    0.8440       196
           2     0.7294    0.6327    0.6776       196

    accuracy                         0.8821      1120
   macro avg     0.8338    0.8116    0.8214      1120
weighted avg     0.8775    0.8821    0.8790      1120



In [47]:
print("\nConfusion Matrix:")
print(restaurant_metrics['confusion_matrix'])


Confusion Matrix:
[[699   6  23]
 [  8 165  23]
 [ 48  24 124]]


```
              precision    recall  f1-score   support

           0       0.91      0.97      0.94       728
           1       0.85      0.85      0.85       196
           2       0.80      0.59      0.68       196

    accuracy                           0.88      1120
   macro avg       0.85      0.80      0.82      1120
weighted avg       0.88      0.88      0.88      1120
```

In [48]:
model.eval()
pred = trainer.predict(ds_laptop_val)

laptop_metrics = compute_metrics(
    (pred.predictions, ds_laptop_val['labels']))
print(laptop_metrics['classification_report'])

/usr/local/lib/python3.13/dist-packages/transformers/tokenization_utils_base.py:2699: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


              precision    recall  f1-score   support

           0     0.9067    0.9120    0.9094       341
           1     0.6452    0.9375    0.7643       128
           2     0.7982    0.5148    0.6259       169

    accuracy                         0.8119       638
   macro avg     0.7833    0.7881    0.7665       638
weighted avg     0.8255    0.8119    0.8052       638



In [ ]:
print("\nConfusion Matrix:")
print(laptop_metrics['confusion_matrix'])

```
              precision    recall  f1-score   support

           0       0.92      0.90      0.91       341
           1       0.72      0.91      0.80       128
           2       0.77      0.64      0.70       169

    accuracy                           0.83       638
   macro avg       0.80      0.82      0.80       638
weighted avg       0.84      0.83      0.83       638
```

In [50]:
model.eval()
pred = trainer.predict(ds_fabsa_test)

fabsa_metrics = compute_metrics(
    (pred.predictions, ds_fabsa_test['labels']))
print(fabsa_metrics['classification_report'])

/usr/local/lib/python3.13/dist-packages/transformers/tokenization_utils_base.py:2699: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


              precision    recall  f1-score   support

           0     0.9622    0.9753    0.9687      1825
           1     0.9383    0.9163    0.9272       896
           2     0.8506    0.8132    0.8315        91

    accuracy                         0.9513      2812
   macro avg     0.9170    0.9016    0.9091      2812
weighted avg     0.9509    0.9513    0.9510      2812



In [ ]:
print("\nConfusion Matrix:")
print(fabsa_metrics['confusion_matrix'])

```
              precision    recall  f1-score   support

           0       0.95      0.94      0.94      1825
           1       0.92      0.79      0.85       896
           2       0.31      0.81      0.44        91

    accuracy                           0.89      2812
   macro avg       0.73      0.85      0.75      2812
weighted avg       0.92      0.89      0.90      2812
```

In [ ]:
# model.push_to_hub(repo_id, token = "hf_XXXX", private=True)

In [ ]:
# tokenizer.push_to_hub(repo_id, token = "hf_XXXX", private=True)